# 🔬 Breast Cancer (BUSI) MoE-Segformer Baseline Trainer

This notebook trains the MoE-SegFormer **BASELINE** on the Breast Ultrasound Images Dataset (BUSI) natively on Google Colab.
We will compare this baseline against the SegMoTE approach.


In [ ]:
# 1. Mount Google Drive to save checkpoints persistently (Optional)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone Repo and Install Dependencies
%cd /content
!rm -rf vision_tranformer_moe
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git
%cd /content/vision_tranformer_moe
# Hotfix: Ensure loss function is moved to GPU device
!sed -i 's/criterion = CombinedSegmentationLoss(loss_type=loss_type, dice_weight=dice_weight)/criterion = CombinedSegmentationLoss(loss_type=loss_type, dice_weight=dice_weight).to(device)/g' training/train_moe.py
!pip install -r requirements.txt
!pip install kaggle albumentations tensorboard transformers torch torchvision

## 3. Authenticate with Kaggle
Using the provided Kaggle API Token to download the dataset seamlessly.

In [ ]:
import os

# Set the provided Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_f92b021c2b42601bd960c76192014a55"

# Ensure the Kaggle CLI uses it
!mkdir -p ~/.kaggle
!echo "KGAT_f92b021c2b42601bd960c76192014a55" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token
print("Kaggle authentication configured!")

In [ ]:
# 4. Download and Prepare the BUSI Dataset
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
!unzip -q breast-ultrasound-images-dataset.zip -d busi_temp

import os
import shutil
import glob

dest_dir = "data/medical_dataset"
os.makedirs(os.path.join(dest_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(dest_dir, "masks"), exist_ok=True)

# The BUSI dataset contains folders: benign, malignant, normal
# Inside each, files are named: <name>.png (image), <name>_mask.png (mask)
src_dir = "busi_temp/Dataset_BUSI_with_GT"
for category in ["benign", "malignant", "normal"]:
    cat_path = os.path.join(src_dir, category)
    if not os.path.exists(cat_path):
        continue
    
    # Get all images (those that don't have 'mask' in the name)
    all_files = glob.glob(os.path.join(cat_path, "*.png"))
    images = [f for f in all_files if "mask" not in f]
    
    for img_path in images:
        base = os.path.splitext(os.path.basename(img_path))[0]
        # Copy image
        shutil.copy(img_path, os.path.join(dest_dir, "images", f"{category}_{base}.png"))
        
        # Copy primary mask
        mask_path = os.path.join(cat_path, f"{base}_mask.png")
        if os.path.exists(mask_path):
            shutil.copy(mask_path, os.path.join(dest_dir, "masks", f"{category}_{base}_mask.png"))

print("BUSI Dataset successfully formatted for the framework!")

In [ ]:
# 5. Configure 40 Epochs for optimal convergence
import yaml
config_path = "configs/moe_segmentation.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['training']['epochs'] = 40
config['dataset']['name'] = 'medical-image-mask'
config['dataset']['data_dir'] = 'data/medical_dataset'
config['dataset']['batch_size'] = 2

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config updated to 40 epochs for baseline.")

In [ ]:
# 6. Launch Baseline Training
!PYTHONPATH=. python training/train_moe.py --config configs/moe_segmentation.yaml

In [ ]:
# 7. Visualize Results
import matplotlib.pyplot as plt
from PIL import Image
import glob
import os

plots = glob.glob("outputs/moe_segmentation/plots/*.png")
if plots:
    for p in sorted(plots):
        print(f"\nDisplaying: {os.path.basename(p)}")
        img = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No plots found. Ensure training successfully ran!")

In [ ]:
# 8. Download Results
!zip -r moe_segmentation_results.zip outputs/moe_segmentation/
from google.colab import files
files.download("moe_segmentation_results.zip")